In [ ]:
# ==============================
# 🤖 Install dependencies
# ==============================
!pip install -q bitsandbytes accelerate transformers

# ==============================
# 🤖 Load Model (CORRECT 4-bit way)
# ==============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

torch.cuda.empty_cache()

model_name = "sarvamai/sarvam-translate"

# 4-bit config (FIX)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config   # ✅ correct way
)

# ==============================
# 🔁 Translation Function
# ==============================
def translate_text(text, tgt_lang="Sanskrit"):
    messages = [
        {"role": "system", "content": f"Translate the text below to {tgt_lang}."},
        {"role": "user", "content": text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return output_text.strip()

# ==============================
# 🔐 Google Sheets Auth
# ==============================
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
from tqdm import tqdm

creds, _ = default()
gc = gspread.authorize(creds)

# ==============================
# 🌐 Open Sheet
# ==============================
spreadsheet_url = "https://docs.google.com/spreadsheets/d/17Z102gEEQakxuEc2t4qLLmXDrFld8Vn-AyHKkv4Vbfo/edit?usp=sharing"

spreadsheet = gc.open_by_url(spreadsheet_url)
worksheet = spreadsheet.worksheet("IAST_1000")

rows = worksheet.get_all_values()
df = pd.DataFrame(rows[1:], columns=rows[0])

print("Preview:")
display(df.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Preview:


,IAST_term,English_meaning,input_text,target_text,ByT5_Output,Ind_ByT5_Combined_Hindi_OP,Ind_ByT5_Sanskrit_OP,NLLB_Devanagari_OP,NLLB_IAST_OP,Indic_Sanskrit_OP,Sarvam_Devanagari_OP,Sarvam_IAST_OP,Qwen_IAST_OP,Qwen_Devanagari_OP,Nalanda_Devanagari_OP,Srvm_op_test,Sanskrit5_Output
0,hikkā,hiccup,symptom: hiccup,hikkā,,हिचकी,,आयुर्वेदिकं लक्षणवर्णनं - हिकुकम् ।,āyurvedikaṃ lakṣaṇavarṇanaṃ - hikukam .,हिचकः ।,संस्कृतभाषायाः संस्कृतसंस्कृतसंस्कृतिं संस्कृत...,saṃskṛtabhāṣāyāḥ saṃskṛtasaṃskṛtasaṃskṛtiṃ saṃ...,kṣudra-kṣumba,क्षुद्र-क्षुम्ब,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",हिक्का,hiccup_hiccup_SNM
1,śvāsa,breathlessness/difficult breathing,symptom: breathlessness/difficult breathing,śvāsa,,सांस फूलना / सांस लेने में कठिनाई,symptom / Sanskrit Hindi,आयुर्वेदिकं लक्षणवर्णनं- श्वासं न श्वासः/ श्वा...,āyurvedikaṃ lakṣaṇavarṇanaṃ- śvāsaṃ na śvāsaḥ/...,श्वासोच्छ्वासस्य अभावः / कष्टप्रदः श्वासोच्छ्वासः,संस्कृतभाषायाः संस्कृतसंस्कृतसंस्कृतिं संस्कृत...,saṃskṛtabhāṣāyāḥ saṃskṛtasaṃskṛtasaṃskṛtiṃ saṃ...,kṣvabhāva,क्ष्वभाव,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",श्वसनकष्टम्/कठिनश्वासः,breathlessness_breathlessness_U __different_U ...
2,cakṣurādīnāmupaghāta,impairment of eye,symptom: impairment of sense organs viz. eye,cakṣurādīnāmupaghāta,,आँखों की हानि,translation,आयुर्वेदिकं लक्षणवर्णनं- नेत्रदोषः ।,āyurvedikaṃ lakṣaṇavarṇanaṃ- netradoṣaḥ .,नेत्रदोषः ।,संस्कृतभाषायाः संस्कृतसंस्कृतसंस्कृतिं संस्कृत...,saṃskṛtabhāṣāyāḥ saṃskṛtasaṃskṛtasaṃskṛtiṃ saṃ...,śūdrāna,शूद्रान,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",नेत्रक्षतिः,impairment_ob_U eye_eye_U
3,pīnasa,"cold, catarrh","symptom: cold, catarrh",pīnasa,,शीत पित्त,translation,"आयुर्वेदिकं लक्षणवर्णनं - सर्दी, कैटार्ह","āyurvedikaṃ lakṣaṇavarṇanaṃ - sardī, kaiṭārha",शीतकालः - शीतकालः - शीतकालः - शीतकालः - शीतकाल...,संस्कृतभाषायाः संस्कृतसंस्कृतसंस्कृतिं संस्कृत...,saṃskṛtabhāṣāyāḥ saṃskṛtasaṃskṛtasaṃskṛtiṃ saṃ...,śirāśira,शिराशिर,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...","शीत, श्लेष्म",cold_cold_U
4,ardita,facial paralysis,symptom: facial paralysis,ardita,,चेहरे का पक्षाघात,symptom,आयुर्वेदिकं लक्षणवर्णनं- अनुहारस्य पक्षाघातः ।,āyurvedikaṃ lakṣaṇavarṇanaṃ- anuhārasya pakṣāg...,मुखस्य पक्षाघातः ।,संस्कृतभाषायाः संस्कृतसंस्कृतसंस्कृतिं संस्कृत...,saṃskṛtabhāṣāyāḥ saṃskṛtasaṃskṛtasaṃskṛtiṃ saṃ...,ardita,अर्दित,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",मुखपक्षाघातः,facial_paralysis_


In [ ]:

# ==============================
# ⚙️ Process FIRST 40 rows
# ==============================
texts = df["English_meaning"].fillna("").tolist()[:40]

results = []

for text in tqdm(texts):
    try:
        out = translate_text(text)
    except:
        out = ""
    results.append(out)

print("Translation complete ✅")

# ==============================
# 🔠 Column Helper
# ==============================
def col_to_letter(col_num):
    result = ""
    while col_num:
        col_num, rem = divmod(col_num - 1, 26)
        result = chr(65 + rem) + result
    return result

# ==============================
# ✍️ Write to Sheet
# ==============================
header = worksheet.row_values(1)

if "Srvm_op_test" not in header:
    worksheet.update_cell(1, len(header) + 1, "Srvm_op_test")
    col_index = len(header) + 1
else:
    col_index = header.index("Srvm_op_test") + 1

col_letter = col_to_letter(col_index)

column_data = ["Srvm_op_test"] + results

cell_range = f"{col_letter}1:{col_letter}{len(column_data)}"
cells = worksheet.range(cell_range)

for i, cell in enumerate(cells):
    cell.value = column_data[i]

worksheet.update_cells(cells)

print("✅ Output saved successfully!")

100%|██████████| 40/40 [01:23<00:00,  2.09s/it]


Translation complete ✅
✅ Output saved successfully!
